# Step 05 — each site measures with its own instrument

**Data type: RNA_array** (GSE65391). **Reads:** `step04_site_{A,B,C}.rds`.
**Writes:** `step05_site_{A,B,C}.rds`.

Real sites differ in laboratory, reagent lot and scanner, and each difference moves every gene a
little. We imitate that: each site draws its own **shift** and **scale** for every gene and applies
them to its own data, and nothing is exchanged.

- **shift**: added to every sample at the site (standard deviation 0.4 on the log2 scale, so a typical
  gene moves by about 30% up or down);
- **scale**: widens or narrows the spread of the gene around the site's own mean (log-normal,
  standard deviation 0.25).

This is the model ComBat assumes, so it is a fair test of the method, not of the model.

**The data before this step stay on disk** (`step04_site_*.rds`). Only oracle cells read them, as the
truth against which corrections are measured in step 08.

In [1]:
source("../src/paths.R")
for (i in seq_along(SITES)) {
  s <- SITES[i]
  d <- readRDS(site_file("04", s))
  set.seed(SEED + 50L + i)                              # this site's instrument
  shift <- rnorm(nrow(d$E), 0, 0.4)
  scale <- rlnorm(nrow(d$E), 0, 0.25)
  mu    <- rowMeans(d$E)
  d$E   <- mu + (d$E - mu) * scale + shift
  d$planted <- data.frame(gene = rownames(d$E), shift = shift, scale = scale)
  saveRDS(d, site_file("05", s))
  cat(sprintf("site %s: %d samples, median |shift| %.2f, scale range %.2f-%.2f\n",
              s, ncol(d$E), median(abs(shift)), min(scale), max(scale)))
}

site A: 338 samples, median |shift| 0.27, scale range 0.33-2.69
site B: 280 samples, median |shift| 0.27, scale range 0.34-2.61
site C: 354 samples, median |shift| 0.27, scale range 0.39-2.71


In [2]:
# the planted shift of three genes at each site
sapply(SITES, function(s) {
  p <- readRDS(site_file("05", s))$planted
  round(setNames(p$shift, p$gene)[c("IFI27", "ISG15", "HBB")], 2)
})

,A,B,C
IFI27,0.58,-0.36,-0.34
ISG15,-0.76,0.43,0.42
HBB,-0.06,0.24,-0.80


## Findings

Every gene at every site now carries a known, site-specific distortion. Step 06 removes it without
moving any sample between sites.